<a href="https://colab.research.google.com/github/barry-clarke/CS5004/blob/main/Etivity3_24325082_Tasks3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
#!pip install gymnasium[atari] ale-py

In [10]:
import gymnasium as gym
from gymnasium.wrappers import AtariPreprocessing, FrameStackObservation
import ale_py
import numpy as np
import tensorflow as tf
from collections import deque

# --- 1. ENVIRONMENT SETUP (MS. PAC-MAN) ---
# Explicitly register the Atari environments
gym.register_envs(ale_py)

# Standard Atari parameters to prepare the environment for wrappers
env = gym.make(
    "ALE/MsPacman-v5",
    frameskip=1,
    repeat_action_probability=0.0,
    full_action_space=False
)

# Wrappers: Grayscale, resize to 84x84, and stack 4 frames together
env = AtariPreprocessing(env, frame_skip=4)
env = FrameStackObservation(env, 4)

n_outputs = int(env.action_space.n)

# --- 2. CNN ARCHITECTURES (MAIN AND TARGET) ---
def create_dqn_model():
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=(4, 84, 84)),   # Accept the shape exactly as the Gymnasium wrapper outputs it
        tf.keras.layers.Permute((2, 3, 1)),         # Swap the axes so the 4 stacked frames move to the channels position
        tf.keras.layers.Rescaling(1.0 / 255.0),     # Normalise pixel values from 0-255 to 0.0-1.0

        tf.keras.layers.Conv2D(32, kernel_size=8, strides=4, activation="relu"),
        tf.keras.layers.Conv2D(64, kernel_size=4, strides=2, activation="relu"),
        tf.keras.layers.Conv2D(64, kernel_size=3, strides=1, activation="relu"),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(512, activation="relu"),
        tf.keras.layers.Dense(n_outputs)
    ])

# Main network (learns every step) and Target network (provides stable references)
model = create_dqn_model()
target_model = create_dqn_model()
target_model.set_weights(model.get_weights())

# --- 3. OPTIMISER AND LOSS ---
optimizer = tf.keras.optimizers.Adam(learning_rate=0.00025, clipnorm=1.0)
loss_fn = tf.keras.losses.Huber()

### Task Requirement: Target Calculation and SGD Update
**Where the target is calculated:**
Within the `@tf.function` training step below, the target calculation occurs at the Bellman equation line: `target_Q_values = rewards + runs * 0.99 * max_next_Q_values`. This combines the immediate reward with the discounted maximum expected future reward (gamma = 0.99) sourced from the stable target network. The `runs` variable acts as a mask to ensure that if the game ends, the target is solely the immediate reward.

**Where the update to the weights is performed:**
This occurs at the end of the `GradientTape` block: `optimizer.apply_gradients(zip(grads, model.trainable_variables))`. Backpropagation calculates the derivative of the Huber loss with respect to the network's weights, and the Adam optimiser applies those gradients to update the model.

In [11]:
# --- 4. THE CORE TRAINING STEP ---
@tf.function
def training_step(states, actions, rewards, next_states, dones):
    # Ask the TARGET model for the next states' Q-values (Standard DQN)
    next_Q_values = target_model(next_states, training=False)
    max_next_Q_values = tf.reduce_max(next_Q_values, axis=1)

    runs = 1.0 - tf.cast(dones, tf.float32)
    rewards = tf.cast(rewards, tf.float32)

    # --- TARGET CALCULATION LOCATED HERE ---
    target_Q_values = rewards + runs * 0.99 * max_next_Q_values
    target_Q_values = tf.reshape(target_Q_values, [-1, 1])
    # ---------------------------------------

    mask = tf.one_hot(actions, n_outputs)

    with tf.GradientTape() as tape:
        all_Q_values = model(states, training=True)
        Q_values = tf.reduce_sum(all_Q_values * mask, axis=1, keepdims=True)
        loss = tf.reduce_mean(loss_fn(target_Q_values, Q_values))

    # --- WEIGHT UPDATE (SGD/ADAM) LOCATED HERE ---
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    # ---------------------------------------------

In [12]:
# --- 5. Output for Verification ---
print("--- ENVIRONMENT DETAILS ---")
print(f"Environment: {env.spec.id}")
print(f"Observation Space (Wrapped): {env.observation_space}")
print(f"Action Space: {env.action_space} (Total Actions: {n_outputs})")

print("\n--- CNN ARCHITECTURE SUMMARY ---")
model.summary()

print("\n--- FORWARD PASS TEST ---")
# Pass a dummy state (1 batch, 4 stacked frames, 84x84 pixels) through the network
dummy_state = np.zeros((1, 4, 84, 84), dtype=np.float32)
dummy_q_values = model(dummy_state)
print(f"Output Q-Values shape: {dummy_q_values.shape}")
print(f"Output Q-Values: {dummy_q_values.numpy()}")

--- ENVIRONMENT DETAILS ---
Environment: ALE/MsPacman-v5
Observation Space (Wrapped): Box(0, 255, (4, 84, 84), uint8)
Action Space: Discrete(9) (Total Actions: 9)

--- CNN ARCHITECTURE SUMMARY ---


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ permute_4 (Permute)             │ (None, 84, 84, 4)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_7 (Rescaling)         │ (None, 84, 84, 4)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_21 (Conv2D)              │ (None, 20, 20, 32)     │         8,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_22 (Conv2D)              │ (None, 9, 9, 64)       │        32,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_23 (Conv2D)              │ (None, 7, 7, 64)       │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_7 (Flatten)             │ (None, 3136)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 512)            │     1,606,144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 9)              │         4,617 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,688,745 (6.44 MB)

 Trainable params: 1,688,745 (6.44 MB)

 Non-trainable params: 0 (0.00 B)


--- FORWARD PASS TEST ---
Output Q-Values shape: (1, 9)
Output Q-Values: [[0. 0. 0. 0. 0. 0. 0. 0. 0.]]


### The Epsilon-Greedy Training Engine
*Note: Training a Ms. Pac-Man DQN from raw pixels requires millions of frames and extensive GPU time. For the purpose of this submission, the training loop below is artificially capped at 5 episodes. This successfully demonstrates that the architecture, memory buffer, and custom training step compile and execute functionally without exceeding local computational constraints.*

In [13]:
import time # Added to track how long the training takes

# --- 6. REPLAY BUFFER ---
batch_size = 32
max_memory_length = 100000
replay_buffer = deque(maxlen=max_memory_length)

def sample_experiences(batch_size):
    """Randomly samples a mini-batch of experiences from memory."""
    indices = np.random.randint(len(replay_buffer), size=batch_size)
    batch = [replay_buffer[index] for index in indices]

    states, actions, rewards, next_states, dones = [
        np.array([experience[field_index] for experience in batch])
        for field_index in range(5)
    ]
    return states, actions, rewards, next_states, dones

# --- 7. ACTION SELECTION ---
def epsilon_greedy_policy(state, epsilon):
    if np.random.rand() < epsilon:
        return np.random.randint(n_outputs)
    else:
        state_tensor = tf.convert_to_tensor(state, dtype=tf.float32)
        Q_values = model(state_tensor[tf.newaxis], training=False)
        return tf.argmax(Q_values[0]).numpy()

# --- 8. MAIN TRAINING LOOP ---
episodes = 5  # Capped for academic submission
epsilon_min = 0.1
epsilon_decay_steps = 1000000.0
update_target_network_steps = 10000
global_step = 0

# New tracking variables for the summary
episode_rewards_history = []
start_time = time.time()

print("Starting training loop...")

for episode in range(episodes):
    obs, info = env.reset()
    state = np.array(obs)
    episode_reward = 0

    while True:
        global_step += 1
        epsilon = max(1.0 - global_step / epsilon_decay_steps, epsilon_min)
        action = epsilon_greedy_policy(state, epsilon)

        next_obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        next_state = np.array(next_obs)

        replay_buffer.append((state, action, reward, next_state, done))
        state = next_state
        episode_reward += reward

        if len(replay_buffer) > batch_size:
            states, actions, rewards, next_states, dones = sample_experiences(batch_size)
            training_step(states, actions, rewards, next_states, dones)

        if global_step % update_target_network_steps == 0:
            target_model.set_weights(model.get_weights())
            print(f"--> Target network synced at step {global_step}")

        if done:
            break

    # Record the reward and update the print statement to show progress (e.g., 1/5)
    episode_rewards_history.append(episode_reward)
    print(f"Episode: {episode + 1}/{episodes} | Reward: {episode_reward} | Epsilon: {epsilon:.3f} | Total Steps: {global_step}")

env.close()

# --- 9. TRAINING SUMMARY ---
elapsed_time = time.time() - start_time
avg_reward = np.mean(episode_rewards_history)
max_reward = np.max(episode_rewards_history)

print("\n" + "="*45)
print("              TRAINING COMPLETE              ")
print("="*45)
print(f"Total Episodes Executed : {episodes}")
print(f"Total Global Steps      : {global_step}")
print(f"Total Time Elapsed      : {elapsed_time:.2f} seconds")
print(f"Average Reward          : {avg_reward:.2f}")
print(f"Maximum Reward Achieved : {max_reward:.1f}")
print(f"Final Epsilon Value     : {epsilon:.3f}")
print(f"Replay Buffer Size      : {len(replay_buffer)} / {max_memory_length}")
print("="*45)

Starting training loop...
Episode: 1/5 | Reward: 180.0 | Epsilon: 0.999 | Total Steps: 508
Episode: 2/5 | Reward: 150.0 | Epsilon: 0.999 | Total Steps: 882
Episode: 3/5 | Reward: 170.0 | Epsilon: 0.999 | Total Steps: 1410
Episode: 4/5 | Reward: 240.0 | Epsilon: 0.998 | Total Steps: 1859
Episode: 5/5 | Reward: 250.0 | Epsilon: 0.998 | Total Steps: 2301

             🏆 TRAINING COMPLETE 🏆             
Total Episodes Executed : 5
Total Global Steps      : 2301
Total Time Elapsed      : 196.56 seconds
Average Reward          : 198.00
Maximum Reward Achieved : 250.0
Final Epsilon Value     : 0.998
Replay Buffer Size      : 2301 / 100000


### Interpretation of Training Results

```text
Starting training loop...
Episode: 1/5 | Reward: 180.0 | Epsilon: 0.999 | Total Steps: 508
Episode: 2/5 | Reward: 150.0 | Epsilon: 0.999 | Total Steps: 882
Episode: 3/5 | Reward: 170.0 | Epsilon: 0.999 | Total Steps: 1410
Episode: 4/5 | Reward: 240.0 | Epsilon: 0.998 | Total Steps: 1859
Episode: 5/5 | Reward: 250.0 | Epsilon: 0.998 | Total Steps: 2301

=============================================
             🏆 TRAINING COMPLETE 🏆             
=============================================
Total Episodes Executed : 5
Total Global Steps      : 2301
Total Time Elapsed      : 196.56 seconds
Average Reward          : 198.00
Maximum Reward Achieved : 250.0
Final Epsilon Value     : 0.998
Replay Buffer Size      : 2301 / 100000
=============================================
```

**1. System Health and Execution Validation**
The output confirms that the underlying architecture of the handcrafted Deep Q-Network is functioning exactly as intended:
* **Experience Replay Integration:** The final Replay Buffer Size perfectly matches the Total Global Steps (2301). This verifies that every frame transition is being correctly processed, packaged into a tuple, and securely stored in memory.
* **Custom Training Step Stability:** Processing 2301 steps without error demonstrates that the custom `@tf.function`, the `GradientTape` backpropagation, and the tensor reshaping within the Bellman equation are handling the 32-sample mini-batches smoothly.
* **Computational Constraints:** Processing just over 2300 frames required approximately 3 minutes and 16 seconds. Extrapolating this performance, reaching the 1,000,000 frames required to complete the epsilon decay would take nearly 24 hours of continuous compute. Noting this metric justifies artificially capping the execution at 5 episodes for the purpose of this e-tivity submission.

**2. Analysis of Agent Behaviour**
While the agent's score increased from 180.0 in Episode 1 to 250.0 in Episode 5, this does not indicate early learning.
* **Epsilon Dominance:** The final epsilon value rests at 0.998. This confirms that for 99.8% of the steps, the agent bypassed the neural network's predictions and selected a purely random action.
* **Random Walk:** The variance in scores is the result of a random walk through the environment. Sometimes the agent randomly navigates into a reward-rich corridor (scoring 250), and other times it immediately wanders into a ghost (scoring 150).
* **Conclusion:** As the training was capped at 5 episodes, the agent remains entirely in the "exploration" phase. Rather than playing strategically, it is moving randomly to fill its memory buffer with basic experiences (eating, moving, dying). True learning and consistently higher scores will only emerge after thousands of episodes, once the random action rate (epsilon) has decayed significantly.

### DQN Improvements

REF: https://apxml.com/courses/intermediate-reinforcement-learning/chapter-3-dqn-improvements-variants

**Resolving Maximisation Bias via Double DQN (DDQN)**
A common flaw in standard DQN is maximisation bias. The network often overestimates the true value of certain states, as the target calculation relies on a strict "max" operation applied to noisy, estimated Q-values. To fix this, the Double DQN (DDQN) improvement separates action selection from action evaluation. The main network chooses the best action, while the separate target network evaluates the actual score of that specific move. This structural change cuts down on overly optimistic guesses and helps to stabilise the learning process.

**Resolving Catastrophic Forgetting via Prioritised Experience Replay (PER)**
The standard Experience Replay buffer helps prevent the agent from forgetting past lessons, but it pulls old memories completely at random. As a result, the agent might repeatedly review uninformative moments while losing track of rare but highly instructive mistakes. Prioritised Experience Replay (PER) solves this by ranking memories based on how inaccurate the network's original prediction was. Memories with larger errors receive a higher priority score. This ensures the agent regularly reviews its biggest mistakes, preventing the catastrophic forgetting of critical edge cases.

**Improving State Evaluation via Dueling Architectures**
In many game states, the choice of action has no real impact on what happens next. Standard DQN calculates a full value for every action regardless of this fact. To improve efficiency, the Dueling architecture splits the neural network's final layers into two separate streams. One stream estimates the overall value of being in a particular state, while the other calculates the specific advantage of taking each available action. These streams are combined at the final output. This allows the agent to learn which states are inherently valuable without having to calculate the exact effect of every single move.

**Accelerating Reward Propagation via Multi-Step Learning**
Standard DQN updates its weights based on the reward from just one single step into the future. This makes learning slow, as the reward from a distant goal takes a long time to propagate backwards through the network's memory. Multi-step learning calculates the target by looking several steps ahead (often three or five) before estimating the final future value. This drastically speeds up the learning process and helps the agent link early actions to delayed rewards much faster.